## Run predictions using the ICEBERG model

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import torch
from tqdm.notebook import tqdm # Use notebook-optimized progress bars
from matchms.importing import load_from_mgf

In [2]:
# --- 1. SETUP MS-PRED ---
# Ensure this points to your ms-pred folder
repo_root = '/data/nas-gpu/wang/tmach007/ms-pred'
sys.path.append('/data/nas-gpu/wang/tmach007/ms-pred')

In [3]:
try:
    import ms_pred.common as common
    from ms_pred.dag_pred.iceberg_elucidation import (
        iceberg_prediction, 
        load_global_config, 
        load_pred_spec
    )
    print("Successfully imported ms_pred modules.")
except ImportError as e:
    print(f"CRITICAL ERROR: {e}")
    sys.exit(1)

Successfully imported ms_pred modules.


In [4]:
# --- 2. HELPER FUNCTIONS ---
def clean_gnps_adduct(adduct):
    if pd.isna(adduct) or not adduct: return '[M+H]+'
    s = str(adduct).strip()
    
    # 1. Handle common GNPS "1+" or "1-" notation
    s = s.replace('1+', '+').replace('1-', '-')
    
    # 2. Standardize element capitalization (e.g., NA -> Na)
    s = s.replace('NA', 'Na').replace('K+', 'K')
    
    # 3. Ensure standard wrapping
    if not s.startswith('['): s = f"[{s}]"
    
    # 4. Correct for common typos found in GNPS batches
    s = s.replace('[M+Na]+', '[M+Na]+')
    
    return s

In [20]:
# --- 3. PREDICTION FUNCTION ---
def predict_single_batch(input_tsv, input_mgf, output_dir, gpu_id=0):
    """
    Function to run ICEBERG prediction on a specific TSV/MGF pair.
    """
    # 1. Configuration Setup
    iceberg_yaml = os.path.join(repo_root, 'configs/iceberg/iceberg_elucidation.yaml')
    os.makedirs(output_dir, exist_ok=True)
    
    device = f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu"
    local_config = load_global_config(iceberg_yaml)
    local_config['device'] = device
    local_config['nce'] = True # GNPS typically uses NCE

    # 2. Load Honest Pairs
    df_honest = pd.read_csv(input_tsv, sep='\t')
    if df_honest.empty:
        print(f"File {input_tsv} is empty.")
        return
    
    unique_gnps_ids = df_honest['gnps_id'].astype(str).unique()
    print(f"Processing {len(unique_gnps_ids)} spectra from {os.path.basename(input_tsv)}")

    # 3. Extract Metadata from MGF
    metadata_lookup = {}
    for spec in tqdm(load_from_mgf(input_mgf), desc="Scanning MGF"):
        sid = str(spec.get("scans") or spec.get("title"))
        if sid in unique_gnps_ids:
            metadata_lookup[sid] = {
                'smiles': spec.get("smiles"),
                'nce': float(spec.get("collision_energy") or 30.0),
                'adduct': clean_gnps_adduct(spec.get("adduct"))
            }

    # 4. Prediction Loop
    results = []
    original_cwd = os.getcwd()
    
    for gnps_id, meta in tqdm(metadata_lookup.items(), desc="ICEBERG Inference"):
        try:
            # Shift to repo root to allow ICEBERG to find internal modules
            os.chdir(repo_root)

            print(f"DEBUG: Predicting GNPS ID {gnps_id} | SMILES: {meta['smiles']} | Adduct: {meta['adduct']} | NCE: {meta['nce']}")
            
            res_path, pmz = iceberg_prediction(
                candidate_smiles=[meta['smiles']],
                collision_energies=[meta['nce']],
                adduct=meta['adduct'],
                force_recompute=True,
                **local_config
            )
            
            _, pred_specs, _ = load_pred_spec(load_dir=res_path, merge_spec=False)
            
            # Revert to notebook directory
            os.chdir(original_cwd)
            
            if len(pred_specs) > 0:
                spec_dict = pred_specs[0]
                target_key = str(float(meta['nce']))
                raw_peaks = spec_dict.get(target_key, spec_dict.get(list(spec_dict.keys())[0]))
                
                peaks = [(float(p[0]), float(p[1])) for p in raw_peaks if p[1] > 0.005]
                
                results.append({
                    'gnps_id': gnps_id,
                    'smiles': meta['smiles'],
                    'precursor_mz': pmz[0] if isinstance(pmz, list) else pmz,
                    'predicted_peaks': peaks,
                    'applied_nce': meta['nce'],
                    'applied_adduct': meta['adduct']
                })
        except Exception as e:
            print(f"Failed for {gnps_id}: {e}")
            os.chdir(original_cwd)

    # 5. Save results
    batch_name = os.path.basename(input_tsv).replace('.tsv', '.pkl')
    save_path = os.path.join(output_dir, f"preds_{batch_name}")
    pd.DataFrame(results).to_pickle(save_path)
    print(f"Done! Results saved to: {save_path}")

In [21]:
# --- 4. EXECUTION CELL ---
# Simply update these paths for the specific file you want to run
predict_single_batch(
    input_tsv = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/gnps_hard_negative_honest_batches/honest_pairs_batch_1.tsv",
    input_mgf = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/spectra_pairs/ALL_GNPS_cleaned/batch_1.mgf",
    output_dir = "./results/gnps_iceberg_predictions",
    gpu_id = 0
)

Processing 117 spectra from honest_pairs_batch_1.tsv


Scanning MGF: 0it [00:00, ?it/s]

ICEBERG Inference:   0%|          | 0/117 [00:00<?, ?it/s]

DEBUG: Predicting GNPS ID 14726 | SMILES: Nc1ncnc2c1ncn2C1CC(O)C(CO)O1 | Adduct: [M+Na]+ | NCE: 30.0
CUDA_VISIBLE_DEVICES=0,1 /data/nas-gpu/wang/tmach007/ms-pred/y/envs/ms-gen/bin/python src/ms_pred/dag_pred/predict_smis.py \
               --batch-size 32 \
               --num-workers 8 \
               --dataset-labels /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/ff148fa79517931ba37915c0c8e8f035/cands_df_iceberg_elucidation.tsv \
               --sparse-out \
               --sparse-k 100 \
               --max-nodes 100 \
               --threshold 0.0 \
               --gen-checkpoint /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_generate.ckpt \
               --inten-checkpoint /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_score.ckpt \
               --save-dir /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/ff148fa79517931ba37915c0c8e8f035 \
               --adduct-shift \
               --gpu


/data/nas-gpu/wang/tmach007/ms-pred/y/envs/ms-gen/lib/python3.8/site-packages/pytorch_lightning/utilities/seed.py:48: LightningDeprecationWarning: `pytorch_lightning.utilities.seed.seed_everything` has been deprecated in v1.8.0 and will be removed in v2.0.0. Please use `lightning_fabric.utilities.seed.seed_everything` instead.
  rank_zero_deprecation(
Global seed set to 42
Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file weights/canopus_iceberg_score.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file weights/canopus_iceberg_generate.ckpt`


...

2026-02-24 22:52:26,161 INFO: 
adduct_shift: true
batch_size: 32
binned_out: false
dataset_labels: /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/d536b50f4e37657e60a2fa478696872f/cands_df_iceberg_elucidation.tsv
dataset_name: null
debug: false
gen_checkpoint: /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_generate.ckpt
gpu: true
inten_checkpoint: /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_score.ckpt
max_nodes: 100
num_bins: 15000
num_workers: 8
save_dir: /data/nas-gpu/wang/tmach007/.cache/ms-pred/iceberg-elucidation/d536b50f4e37657e60a2fa478696872f
seed: 42
sparse_k: 100
sparse_out: true
split_name: split_22.tsv
subset_datasets: none
threshold: 0.0
upper_limit: 1500

2026-02-24 22:52:26,260 INFO: Loaded gen / inten models from /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_generate.ckpt & /data/nas-gpu/wang/tmach007/ms-pred/weights/canopus_iceberg_score.ckpt
2026-02-24 22:52:26,261 INFO: Preparing entries


  0%|          | 0/1 [00:00<?, ?it/s]

2026-02-24 22:52:30,176 INFO: There are 1 entries to process


100%|██████████| 1/1 [00:00<00:00,  1.21it/s]


Done! Results saved to: ./results/gnps_iceberg_predictions/preds_honest_pairs_batch_1.pkl
